# AI-Powered Semantic Resume Intelligence System: Embedding Training Pipeline
## Flagship Production-Grade Semantic Retrieval Training Engine

This Google Colab notebook implements the training pipeline for **Component 1** of the Semantic Resume Intelligence System: a custom-trained Sentence Transformer model that learns joint representations of candidate resumes, job descriptions, specific roles, and skill profiles.

### System Architecture Overview
The overall Semantic Resume Intelligence System consists of two core components:
1. **Component 1 (Trained here)**: A custom fine-tuned Sentence Transformer (`sentence-transformers/all-mpnet-base-v2`) trained via an InfoNCE-style Symmetric Multiple Negatives Ranking Loss (MNRL) on positive pairs representing alignments (Resume $\leftrightarrow$ JD, Resume $\leftrightarrow$ Job Title, Resume $\leftrightarrow$ Skills).
2. **Component 2 (LLM; Not trained here)**: A lightweight instruction-tuned LLM (e.g., Gemma 3 1B/4B, Phi-3 Mini, Qwen 2.5 Instruct) integrated into the Streamlit application to parse results and generate explainable ATS reports via Retrieval-Augmented Generation (RAG).

This notebook focuses **exclusively** on training the semantic retrieval model (Component 1). It avoids the fragile and mismatch-prone Trainer API configurations, utilizing a **completely custom, deterministic, mixed-precision PyTorch training loop**.

```mermaid
graph TD
    A[Raw Data: Resumes & JDs] --> B[Data Preprocessing & Standard Cleaning]
    B --> C[Dataset Splitting: Document Level]
    C --> D[Positive Pairs Generation: Resume-JD, Role, Skills]
    D --> E[Custom PyTorch Training Loop: Symmetric MNRL]
    E --> F[Validation Evaluation: MRR & Recall@K]
    F --> G[Visualizations: Heatmap & PCA Projections]
    G --> H[Model Export & HF Hub Upload]
```

> [!IMPORTANT]
> **Developer Note on Version Stability**: This notebook pins exact library dependencies and overrides potential conflicts to guarantee stable runs in a standard Google Colab Free environment. It bypasses `SentenceTransformerTrainer` to avoid import errors and numpy binary incompatibilities.

In [1]:
# ============================================================
# 1. ENVIRONMENT SETUP & DEPENDENCY MANAGEMENT
# ============================================================
# Install pinned, stable packages to avoid dependency conflicts
# in Colab. We do not reinstall torch or numpy to protect environment stability.

!pip install -q \
  sentence-transformers==2.7.0 \
  umap-learn==0.5.6 \
  huggingface-hub==0.23.4 \
  scikit-learn==1.3.2 \
  pandas==2.1.4 \
  numpy==1.26.4 \
  matplotlib==3.8.2 \
  plotly==5.18.0 \
  seaborn==0.13.0

print("Environment setup complete. Libraries installed.")

Environment setup complete. Libraries installed.


## Module 2: Configuration & Seeding for Reproducibility

To ensure that training is fully deterministic, we initialize seeds across all active random number generators (Python `random`, `numpy`, and PyTorch's CPU/GPU backend configurations). We also define hyperparameter defaults and locate GPU specifications.

In [2]:
# ============================================================
# 2. CONFIGURATION & REPRODUCIBILITY CONFIG
# ============================================================
import os
import re
import time
import random
import unicodedata
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.amp as amp
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from huggingface_hub import HfApi, login
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount('/content/drive/')
# Global configuration values
MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"
BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_SEQ_LENGTH = 256
DATA_DIR = "/content/drive/MyDrive/SemanticResumeATS/"
MODEL_OUTPUT_DIR = os.path.join(DATA_DIR, "trained_model")
SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Sets seeds across Python random, numpy, and PyTorch for deterministic output.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# GPU detection and VRAM configurations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Target Execution Device: {device}")
if device.type == "cuda":
    print(f"GPU Model Name: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Mounted at /content/drive/
Target Execution Device: cuda
GPU Model Name: Tesla T4
Available GPU Memory: 14.56 GB


## Module 3: Fail-safe Data Loading

This section automatically mounts Google Drive and scans for filetypes (`CSV`, `JSON`, `JSONL`, `Parquet`) containing resumes, jobs, and skills.

> [!NOTE]
> If the notebook is executed on an environment without Google Drive or missing datasets, a clear warning is printed, and a high-quality **Synthetic Dataset Generator** is initialized so that the entire pipeline can execute end-to-end for validation.

In [3]:
# ============================================================
# 3. DATA LOADING & DATASET MANAGEMENT
# ============================================================

import os
import glob
import pandas as pd
from datasets import load_dataset

# ------------------------------------------------------------
# Google Drive Paths
# ------------------------------------------------------------

DATA_DIR = "/content/drive/MyDrive/SemanticResumeATS"

RESUME_DIR = os.path.join(DATA_DIR, "datasets", "resumes")
JOB_DIR = os.path.join(DATA_DIR, "datasets", "jobs")

In [4]:
# ------------------------------------------------------------
# Helper Functions
# ------------------------------------------------------------

def find_file(folder, extensions):
    """
    Returns the first file with the given extension.
    """
    for ext in extensions:
        files = glob.glob(os.path.join(folder, ext))
        if len(files):
            return files[0]
    return None

In [5]:
def load_resume_dataset():

    print("="*60)
    print("Loading Resume Dataset...")
    print("="*60)

    txt_file = find_file(RESUME_DIR, ["*.txt"])
    csv_file = find_file(RESUME_DIR, ["*.csv"])

    # ----------------------------------------------------
    # CSV
    # ----------------------------------------------------

    if csv_file is not None:
        resume_df = pd.read_csv(csv_file)
        print("Resume CSV Loaded")
        print(resume_df.shape)
        return resume_df

    # ----------------------------------------------------
    # TXT Resume Corpus
    # ----------------------------------------------------

    elif txt_file is not None:
        resumes = []
        with open(txt_file, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()

                if len(line) < 50:
                    continue

                parts = line.split(":::")

                if len(parts) != 3:
                    continue

                source = parts[0]
                keywords = parts[1]
                resume_text = parts[2]
                resumes.append({"source": source, "keywords": keywords, "resume_text": resume_text})

        resume_df = pd.DataFrame(resumes)
        print("Resume TXT Loaded")
        print(resume_df.shape)
        return resume_df

    else:

        raise FileNotFoundError(f"Resume dataset not found inside:\n{RESUME_DIR}")

In [6]:
# ============================================================
# LOAD JOB DATASET
# ============================================================

print("="*60)
print("Loading Job Dataset...")
print("="*60)

job_path="/content/drive/MyDrive/SemanticResumeATS/datasets/jobs/job_descriptions.csv"
job_df=pd.read_csv(job_path)
print("Original shape:",job_df.shape)
print("\nColumns:")
print(job_df.columns)
job_df=job_df.rename(columns={"Job Title":"job_title", "Job Description":"job_description"})

job_df=job_df.drop_duplicates()
job_df=job_df.dropna(subset=["job_title","job_description"])
job_df=job_df.reset_index(drop=True)
print("\nFinal shape:",job_df.shape)

Loading Job Dataset...
Original shape: (1615940, 23)

Columns:
Index(['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location',
       'Country', 'latitude', 'longitude', 'Work Type', 'Company Size',
       'Job Posting Date', 'Preference', 'Contact Person', 'Contact',
       'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits',
       'skills', 'Responsibilities', 'Company', 'Company Profile'],
      dtype='object')

Final shape: (1615940, 23)


In [7]:
print(job_df.columns.tolist())

['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location', 'Country', 'latitude', 'longitude', 'Work Type', 'Company Size', 'Job Posting Date', 'Preference', 'Contact Person', 'Contact', 'job_title', 'Role', 'Job Portal', 'job_description', 'Benefits', 'skills', 'Responsibilities', 'Company', 'Company Profile']


In [8]:
# ============================================================
# ESCO KNOWLEDGE BASE
# ============================================================

from datasets import load_dataset
def build_esco_knowledge_base():

    print("="*60)
    print("Building ESCO Knowledge Base...")
    print("="*60)

    datasets_to_try = ["jjzha/esco", "mw4/esco-skills", "TechWolf/ESCO-Skills"]

    for ds in datasets_to_try:
        try:
            print(f"Trying HuggingFace dataset : {ds}")
            data = load_dataset(ds)
            print(f"Successfully loaded {ds}")
            return data

        except Exception:
            pass

    print("No online ESCO dataset found.")
    print("Loading built-in ESCO knowledge base...")

    esco = {
        "python developer":{
            "skills":["python","django","flask","fastapi","sql","postgresql","git","docker","linux","rest api"]},

        "java developer":{
            "skills":["java","spring","spring boot","hibernate","mysql","docker","git","maven","microservices"]},

        ".net developer":{
            "skills":["c#",".net","asp.net","entity framework","sql server","azure","git"]},

        "frontend developer":{
            "skills":["html","css","javascript","react","angular","vue","typescript","git"]},

        "backend developer":{
            "skills":["python","java","node.js","sql","docker","kubernetes","redis","rest api"]},

        "full stack developer":{
            "skills":["html","css","javascript","react","node.js","python","sql","docker","git"]},

        "software engineer":{
            "skills":["python","java","c++","git","oop","algorithms","data structures","linux"]},

        "software developer":{
            "skills":["python","java","c#","sql","git","docker","rest api"]},

        "web developer":{
            "skills":["html","css","javascript","react","node.js","bootstrap","git"]},

        "mobile application developer":{
            "skills":["flutter","dart","android","ios","firebase","git"]},

        "android developer":{
            "skills":["java","kotlin","android sdk","firebase","xml","git"]},

        "ios developer":{
            "skills":["swift","swiftui","xcode","ios sdk","git"]},

        "react developer":{
            "skills":["react","javascript","redux","html","css","typescript"]},

        "angular developer":{
            "skills":["angular","typescript","rxjs","html","css"]},

        "vue developer":{
            "skills":["vue","javascript","vuex","html","css"]},

        "node.js developer":{
            "skills":["node.js","express","mongodb","jwt","rest api","docker"]},

        "php developer":{
            "skills":["php","laravel","mysql","apache","rest api"]},

        "laravel developer":{
            "skills":["php","laravel","mysql","redis","docker"]},

        "wordpress developer":{
            "skills":["wordpress","php","mysql","css","javascript"]},

        "devops engineer":{
            "skills":["docker","kubernetes","terraform","jenkins","linux","aws","bash","ci/cd"]},

        "site reliability engineer":{
            "skills":["linux","docker","kubernetes","terraform","prometheus","grafana","aws"]},

        "cloud engineer":{
            "skills":["aws","azure","gcp","docker","terraform","linux","networking"]},

        "cloud architect":{
            "skills":["aws","azure","gcp","terraform","kubernetes","security","architecture"]},

        "aws engineer":{
            "skills":["aws","ec2","lambda","s3","iam","cloudformation"]},

        "azure engineer":{
            "skills":["azure","azure devops","aks","azure functions","terraform"]},

        "data scientist":{
            "skills":["python","pandas","numpy","statistics","machine learning","sql","matplotlib","scikit-learn"]},

        "data analyst":{
            "skills":["sql","excel","power bi","tableau","python","statistics","pandas"]},

        "business analyst":{
            "skills":["sql","excel","power bi","requirements gathering","jira","agile"]},

        "business intelligence developer":{
            "skills":["power bi","tableau","sql","data warehouse","etl"]},

        "data engineer":{
            "skills":["python","sql","spark","hadoop","airflow","etl","snowflake"]},

        "machine learning engineer":{
            "skills":["python","tensorflow","pytorch","scikit-learn","docker","mlops","aws"]},

        "deep learning engineer":{
            "skills":["tensorflow","keras","pytorch","cnn","rnn","transformers","gpu"]},

        "computer vision engineer":{
            "skills":["opencv","python","cnn","yolo","tensorflow","pytorch"]},

        "nlp engineer":{
            "skills":["python","transformers","bert","huggingface","spacy","nltk","llm"]},

        "ai engineer":{
            "skills":["python","llm","rag","langchain","vector database","transformers","huggingface"]},

        "prompt engineer":{
            "skills":["prompt engineering","llm","openai","gemini","claude","rag"]},

        "mlops engineer":{
            "skills":["mlflow","docker","kubernetes","airflow","aws","python","git"]},

        "research scientist":{
            "skills":["python","deep learning","statistics","research","transformers","pytorch"]},

        "database administrator":{
            "skills":["sql","oracle","mysql","postgresql","backup","replication","performance tuning"]},

        "sql developer":{
            "skills":["sql","stored procedures","triggers","mysql","postgresql","sql server"]},

        "mongodb developer":{
            "skills":["mongodb","nosql","aggregation","replication","node.js"]},

        "oracle developer":{
            "skills":["oracle","pl/sql","sql","database design"]},

        "cyber security analyst":{
            "skills":["network security","wireshark","siem","firewall","incident response","owasp"]},

        "penetration tester":{
            "skills":["kali linux","metasploit","burp suite","owasp","python"]},

        "security engineer":{
            "skills":["iam","siem","network security","linux","aws","azure"]},

        "network engineer":{
            "skills":["tcp/ip","routing","switching","firewall","cisco","vpn"]},

        "system administrator":{
            "skills":["linux","windows server","active directory","bash","powershell"]},

        "embedded engineer":{
            "skills":["c","c++","embedded systems","microcontrollers","rtos","arm"]},

        "robotics engineer":{
            "skills":["ros","python","c++","opencv","robotics","slam"]},

        "iot engineer":{
            "skills":["mqtt","arduino","raspberry pi","embedded c","python"]},

        "blockchain developer":{
            "skills":["solidity","ethereum","smart contracts","web3","javascript"]},

        "game developer":{
            "skills":["unity","unreal engine","c#","c++","game physics"]},

        "qa engineer":{
            "skills":["selenium","manual testing","automation","jira","testng"]},

        "test automation engineer":{
            "skills":["selenium","python","java","cypress","playwright"]},

        "ui ux designer":{
            "skills":["figma","adobe xd","wireframing","prototyping","user research"]},

        "product manager":{
            "skills":["agile","scrum","jira","roadmap","stakeholder management"]},

        "technical product manager":{
            "skills":["agile","scrum","sql","analytics","roadmap","jira"]},

        "project manager":{
            "skills":["agile","scrum","jira","risk management","planning"]},

        "scrum master":{
            "skills":["scrum","agile","jira","facilitation","sprint planning"]},

        "salesforce developer":{
            "skills":["salesforce","apex","lightning","soql","crm"]},

        "sap consultant":{
            "skills":["sap","abap","hana","erp","business process"]},

        "etl developer":{
            "skills":["etl","ssis","informatica","sql","data warehouse"]},

        "big data engineer":{
            "skills":["spark","hadoop","kafka","hive","scala","python"]},

        "data warehouse engineer":{
            "skills":["snowflake","redshift","bigquery","sql","etl"]},

        "linux administrator":{
            "skills":["linux","bash","shell scripting","systemd","networking"]},

        "windows administrator":{
            "skills":["windows server","active directory","powershell","dns","dhcp"]},

        "ethical hacker":{"skills":["kali linux","metasploit","burp suite","owasp","nmap","wireshark","python"]},

        "security analyst":{"skills":["siem","splunk","firewall","wireshark","incident response","network security"]},

        "soc analyst":{"skills":["splunk","siem","threat intelligence","incident response","linux","firewall"]},

        "cloud security engineer":{"skills":["aws","azure","iam","terraform","network security","docker","kubernetes"]},

        "forensics analyst":{"skills":["autopsy","volatility","wireshark","memory analysis","disk forensics","linux"]},

        "data architect":{"skills":["sql","data warehouse","snowflake","etl","data modeling","bigquery"]},

        "data visualization engineer":{"skills":["power bi","tableau","sql","python","matplotlib","analytics"]},

        "gen ai engineer":{"skills":["python","llm","langchain","rag","transformers","huggingface","vector database"]},

        "llm engineer":{"skills":["python","transformers","langchain","rag","prompt engineering","huggingface"]},

        "ai researcher":{"skills":["python","pytorch","tensorflow","transformers","deep learning","statistics"]},

        "prompt engineer":{"skills":["prompt engineering","openai","gemini","claude","rag","langchain"]},

        "rag engineer":{"skills":["langchain","faiss","chroma","vector database","python","llm"]},

        "data annotation specialist":{"skills":["labeling","data cleaning","quality assurance","excel","python"]},

        "seo specialist":{"skills":["seo","google analytics","google ads","content marketing","keyword research"]},

        "digital marketing manager":{"skills":["seo","sem","google analytics","google ads","social media","content marketing","email marketing","marketing automation"]},

        "content writer":{"skills":["copywriting","seo","content marketing","research","editing"]},

        "social media manager":{"skills":["instagram marketing","facebook ads","analytics","content creation","canva"]},

        "hr manager":{"skills":["recruitment","employee engagement","payroll","hrms","communication"]},

        "recruiter":{"skills":["recruitment","linkedin","screening","interviewing","communication"]},

        "financial analyst":{"skills":["excel","financial modeling","sql","power bi","forecasting"]},

        "accountant":{"skills":["tally","excel","bookkeeping","gst","financial reporting"]},

        "operations manager":{"skills":["operations","planning","excel","communication","leadership"]},

        "technical writer":{"skills":["documentation","api documentation","markdown","git","communication"]},

        "support engineer":{"skills":["linux","sql","networking","troubleshooting","customer support"]},

        "system engineer":{"skills":["linux","networking","docker","bash","monitoring"]},

        "database engineer":{"skills":["mysql","postgresql","sql","performance tuning","replication"]},

        "crm developer":{"skills":["salesforce","crm","apex","soql","lightning"]},

        "erp consultant":{"skills":["sap","erp","hana","abap","business process"]},

        "qa analyst":{"skills":["manual testing","selenium","jira","test cases","automation"]},

        "automation engineer":{"skills":["selenium","playwright","python","java","testng"]},

        "gis analyst":{"skills":["arcgis","qgis","python","spatial analysis","geospatial data"]},

        "oracle database administrator":{"skills":["oracle","pl/sql","sql","rman","dataguard","linux","oracle 11g","oracle 12c","performance tuning","backup"]},

        "sql database administrator":{"skills":["sql server","t-sql","ssis","ssrs","stored procedures","database administration","backup","clustering"]},

        "database administrator":{"skills":["sql","oracle","mysql","postgresql","database design","backup","replication","performance tuning"]},

        "database developer":{"skills":["sql","pl/sql","stored procedures","triggers","mysql","postgresql","database design"]},

        "database specialist":{"skills":["sql","oracle","mysql","database administration","backup","performance tuning"]},

        "database engineer":{"skills":["sql","postgresql","mysql","oracle","etl","data modeling","database design"]},

        "etl developer":{"skills":["etl","ssis","informatica","sql","data warehouse","datastage","talend"]},

        "etl engineer":{"skills":["etl","ssis","informatica","airflow","sql","data warehouse","python"]},

        "data warehouse developer":{"skills":["sql","etl","ssis","snowflake","redshift","data warehouse","postgresql"]},

        "data warehouse engineer":{"skills":["snowflake","redshift","bigquery","sql","etl","airflow"]},

        "business intelligence developer":{"skills":["power bi","tableau","sql","ssis","ssrs","data warehouse","etl"]},

        "business intelligence analyst":{"skills":["power bi","tableau","sql","excel","data visualization","analytics"]},

        "report developer":{"skills":["power bi","tableau","ssrs","sql","excel","analytics"]},

        "amazon redshift administrator":{"skills":["redshift","postgresql","sql","aws","etl","data warehouse"]},

        "postgresql administrator":{"skills":["postgresql","sql","backup","replication","performance tuning","linux"]},

        "mysql administrator":{"skills":["mysql","sql","backup","replication","linux","performance tuning"]},

        "mongodb administrator":{"skills":["mongodb","nosql","replication","sharding","backup","linux"]},

        "oracle developer":{"skills":["oracle","pl/sql","sql","forms","reports","database design"]},

        "sql developer":{"skills":["sql","stored procedures","triggers","views","functions","sql server"]},

        "ssis developer":{"skills":["ssis","sql server","etl","data warehouse","stored procedures"]},

        "ssrs developer":{"skills":["ssrs","sql server","reporting","sql","power bi"]},

        "data modeler":{"skills":["data modeling","sql","er diagrams","database design","etl"]},

        "data architect":{"skills":["data modeling","sql","data warehouse","snowflake","aws","etl"]},

        "data governance engineer":{
            "skills":["data governance","sql","data quality","metadata","collibra","python"]},

        "data quality analyst":{
            "skills":["sql","data validation","excel","python","quality assurance","power bi"]},

        "data migration engineer":{
            "skills":["sql","etl","oracle","postgresql","mysql","data migration"]},

        "power bi developer":{
            "skills":["power bi","dax","sql","power query","excel","data modeling"]},

        "tableau developer":{
            "skills":["tableau","sql","data visualization","excel","analytics"]},

        "snowflake developer":{
            "skills":["snowflake","sql","etl","python","data warehouse","airflow"]},

        "bigquery engineer":{
            "skills":["bigquery","sql","gcp","etl","python","data warehouse"]},

        "apache spark developer":{
            "skills":["spark","scala","python","hadoop","sql","kafka"]},

        "kafka engineer":{
            "skills":["kafka","zookeeper","java","spark","stream processing","docker"]},

        "airflow engineer":{
            "skills":["apache airflow","python","sql","etl","docker","data pipeline"]},

        "computer vision researcher":{
            "skills":["opencv","python","cnn","yolo","tensorflow","pytorch","research"]},

        "reinforcement learning engineer":{
            "skills":["python","pytorch","gym","reinforcement learning","deep learning"]},

        "generative ai engineer":{
            "skills":["python","llm","rag","langchain","transformers","huggingface","vector database"]},

        "chatbot developer":{
            "skills":["python","langchain","llm","rag","fastapi","vector database"]},

        "streamlit developer":{
            "skills":["python","streamlit","pandas","plotly","sql","machine learning"]},

        "computer vision developer":{
            "skills":["opencv","python","yolo","tensorflow","pytorch","image processing"]},

        "tensorflow developer":{
            "skills":["tensorflow","keras","python","deep learning","cnn","gpu"]},

        "pytorch developer":{
            "skills":["pytorch","python","deep learning","transformers","cnn"]},

        "data labeling engineer":{
            "skills":["data annotation","python","quality assurance","excel","labeling"]},

        "api developer":{
            "skills":["fastapi","flask","django","rest api","python","jwt"]},

        "microservices developer":{
            "skills":["java","spring boot","docker","kubernetes","microservices","redis"]},

        "golang developer":{
            "skills":["go","grpc","docker","postgresql","microservices"]},

        "rust developer":{
            "skills":["rust","actix","tokio","postgresql","docker"]},

        "c++ developer":{
            "skills":["c++","oop","stl","algorithms","linux","cmake"]},

        "c developer":{
            "skills":["c","pointers","memory management","linux","embedded systems"]},

        "firmware engineer":{
            "skills":["c","c++","embedded systems","rtos","microcontrollers","uart"]},

        "linux kernel engineer":{
            "skills":["linux","c","kernel programming","device drivers","bash"]},

        "network administrator":{
            "skills":["tcp/ip","routing","switching","dns","dhcp","firewall"]},

        "cloud administrator":{
            "skills":["aws","azure","linux","terraform","networking","docker"]},

        "azure architect":{
            "skills":["azure","terraform","kubernetes","network security","architecture"]},

        "aws architect":{
            "skills":["aws","ec2","lambda","terraform","cloudformation","architecture"]},

        "gcp engineer":{
            "skills":["gcp","bigquery","cloud functions","docker","terraform"]},

        "incident responder":{
            "skills":["incident response","siem","splunk","linux","wireshark"]},

        "malware analyst":{
            "skills":["ida pro","ghidra","python","reverse engineering","malware analysis"]},

        "reverse engineer":{
            "skills":["ghidra","ida pro","assembly","c++","reverse engineering"]},

        "vulnerability analyst":{
            "skills":["owasp","burp suite","nmap","wireshark","metasploit"]},

        "seo analyst":{
            "skills":["seo","google analytics","keyword research","content marketing"]},

        "performance marketer":{
            "skills":["google ads","facebook ads","analytics","seo","marketing automation"]},

        "marketing analyst":{
            "skills":["excel","google analytics","power bi","sql","analytics"]},

        "video editor":{
            "skills":["premiere pro","after effects","davinci resolve","video editing"]},

        "graphic designer":{
            "skills":["photoshop","illustrator","figma","canva","branding"]},

        "vfx artist":{
            "skills":["after effects","blender","maya","cinema 4d","compositing"]},

        "ui designer":{
            "skills":["figma","wireframing","prototyping","adobe xd","design systems"]},

        "ux researcher":{
            "skills":["user research","figma","wireframing","prototyping","analytics"]}

        }

    rows = []

    for role,data in esco.items():

      skills=sorted(list(set(data["skills"])))

      rows.append({
          "occupation":role,
          "role":role,
          "skills":skills,
          "skill_string":" ".join(skills)
      })

    esco_df=pd.DataFrame(rows)


    print(f"Built ESCO Knowledge Base with {len(esco_df)} occupations.")

    return esco, esco_df


esco, esco_df = build_esco_knowledge_base()

Building ESCO Knowledge Base...
Trying HuggingFace dataset : jjzha/esco
Trying HuggingFace dataset : mw4/esco-skills
Trying HuggingFace dataset : TechWolf/ESCO-Skills
No online ESCO dataset found.
Loading built-in ESCO knowledge base...
Built ESCO Knowledge Base with 155 occupations.


In [9]:
print(type(esco_df))
print("esco" in globals())

<class 'pandas.core.frame.DataFrame'>
True


In [10]:
# ------------------------------------------------------------
# Execute Loading
# ------------------------------------------------------------

resume_df = load_resume_dataset()
print()

print("="*60)
print("SUMMARY")
print("="*60)

print("Resume Dataset :", resume_df.shape)
print("Job Dataset    :", job_df.shape)
print("ESCO Dataset   :", esco_df.shape)

Loading Resume Dataset...
Resume TXT Loaded
(29780, 3)

SUMMARY
Resume Dataset : (29780, 3)
Job Dataset    : (1615940, 23)
ESCO Dataset   : (155, 4)


## Module 4: Modular Preprocessing Pipeline

Standardizes, cleans, and structures textual data:
- Unicode NFKC normalization and whitespace standardization.
- Lowercasing and email/URL filtering.
- Strip seniority keywords (Sr., Lead, Associate) to standardize role title comparison.
- Regex-based technical skill extraction from resumes.

In [11]:
# ============================================================
# 4. ADVANCED PREPROCESSING PIPELINE
# ============================================================
#
# This block performs:
#
# 1. Resume Cleaning
# 2. Job Description Cleaning
# 3. ESCO Cleaning
# 4. Synonym Normalization
# 5. Role Normalization
# 6. Experience Extraction
# 7. Education Extraction
# 8. Certification Extraction
# 9. Resume Section Extraction
# 10. Regex Skill Extraction
#
# Output:
#
# resumes_clean
# jobs_clean
# esco_df
#
# ============================================================

print("="*60)
print("PREPROCESSING DATASETS")
print("="*60)

import html
import unicodedata

# ============================================================
# TECHNICAL SYNONYM NORMALIZATION
# ============================================================

SYNONYM_MAP = {

    "ml":"machine learning",
    "ai":"artificial intelligence",
    "dl":"deep learning",
    "nlp":"natural language processing",
    "cv":"computer vision",
    "js":"javascript",
    "ts":"typescript",
    "tf":"tensorflow",
    "postgres":"postgresql",
    "mongo":"mongodb",
    "node":"node.js",
    "py":"python",
    "dotnet":".net",
    "asp net":"asp.net",
    "restful":"rest api",
    "ci cd":"ci/cd",
    "aws cloud":"aws",
    "azure cloud":"azure",
    "gcp cloud":"gcp",
    "powerbi":"power bi",
    "scikit learn":"scikit-learn",
    "hugging face":"huggingface"
}

# ============================================================
# ROLE NORMALIZATION
# ============================================================

ROLE_MAP = {
    "software developer":"software engineer",
    "software programmer":"software engineer",
    "python engineer":"python developer",
    "java engineer":"java developer",
    "backend engineer":"backend developer",
    "frontend engineer":"frontend developer",
    "fullstack developer":"full stack developer",
    "fullstack engineer":"full stack developer",
    "ml engineer":"machine learning engineer",
    "ai developer":"ai engineer",
    "data science engineer":"data scientist",
    "dev ops engineer":"devops engineer",
    "site reliability":"site reliability engineer",
    "business intelligence analyst":"business intelligence developer"
}

# ============================================================
# UNICODE NORMALIZATION
# ============================================================

def unicode_normalize(text):

    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    return text

# ============================================================
# BASIC TEXT CLEANING
# ============================================================

def clean_text(text):

    if pd.isna(text):
        return ""

    text = unicode_normalize(text)
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r"http\\S+"," ",text)
    text = re.sub(r"www\\.\\S+"," ",text)
    text = re.sub(r"\\S+@\\S+"," ",text)
    text = re.sub(r"\\n"," ",text)
    text = re.sub(r"\\t"," ",text)
    text = re.sub(r"[^a-z0-9+#./ ]"," ",text)
    text = re.sub(r"\\s+"," ",text)

    return text.strip()

# ============================================================
# SYNONYM NORMALIZATION
# ============================================================

def normalize_synonyms(text):
    """
    Normalize common abbreviations without corrupting words.
    Uses regex word-boundary matching.
    """

    if pd.isna(text):
        return ""

    text=str(text).lower()

    synonym_map={
        "ml":"machine learning",
        "ai":"artificial intelligence",
        "dl":"deep learning",
        "nlp":"natural language processing",
        "cv":"computer vision",
        "js":"javascript",
        "nodejs":"node.js",
        "tf":"tensorflow",
        "pytorch lightning":"pytorch"
    }

    for old,new in synonym_map.items():

        pattern=rf'(?<![a-zA-Z0-9]){re.escape(old)}(?![a-zA-Z0-9])'

        text=re.sub(pattern,new,text)

    return text

# ============================================================
# ROLE NORMALIZATION
# ============================================================

def normalize_role(role):
    if pd.isna(role):
        return ""

    role = clean_text(role)
    role = normalize_synonyms(role)
    if role in ROLE_MAP:
        return ROLE_MAP[role]
    return role

# ============================================================
# EXPERIENCE EXTRACTION
# ============================================================

def extract_experience(text):

    if pd.isna(text):
        return None
    text=str(text).lower()

    patterns=[r'(\d+)\+?\s*(?:years?|yrs?)\s+of\s+experience',
        r'experience\s*[:\-]?\s*(\d+)\+?\s*(?:years?|yrs?)',
        r'minimum\s+(\d+)\+?\s*(?:years?|yrs?)',
        r'at least\s+(\d+)\+?\s*(?:years?|yrs?)',
        r'over\s+(\d+)\+?\s*(?:years?|yrs?)']

    for p in patterns:
        m=re.search(p,text)
        if m:
            return int(m.group(1))

    return None

# ============================================================
# SECTION EXTRACTION
# ============================================================

SECTION_HEADERS={
    "skills":["skills","technical skills","technologies","core competencies"],
    "education":["education","academic","qualification"],
    "experience":["experience","work experience","employment history"],
    "projects":["projects","project experience"],
    "certifications":["certifications","certificate"],
    "summary":["summary","profile","objective","professional summary"]
}

def extract_sections(text):
    if pd.isna(text):
        return {}

    text=str(text)
    sections={}
    lower=text.lower()
    for section,headers in SECTION_HEADERS.items():

        sections[section]=""
        for h in headers:
            idx=lower.find(h)
            if idx!=-1:
                end=len(text)
                for other_headers in SECTION_HEADERS.values():
                    for other in other_headers:
                        j=lower.find(other,idx+len(h))
                        if j!=-1 and j<end:
                            end=j
                sections[section]=text[idx:end].strip()
                break

    return sections


# ============================================================
# BUILD SKILL VOCABULARY
# ============================================================

skill_vocab = set()

for skills in esco_df["skills"]:

    if isinstance(skills, list):

        for skill in skills:

            skill = str(skill).lower().strip()

            if skill:

                skill_vocab.add(skill)

print("Skill Vocabulary Size:", len(skill_vocab))

# ============================================================
# SKILL EXTRACTION
# ============================================================

def extract_skills(text):
    """
    Extract technical skills using regex word-boundary matching.
    Prevents false matches like:
        c -> architecture
        ios -> comparison
        r -> engineer
    """

    if pd.isna(text):
        return []
    text=str(text).lower()
    found=set()
    for skill in skill_vocab:
        skill=skill.lower().strip()
        pattern=rf'(?<![a-zA-Z0-9]){re.escape(skill)}(?![a-zA-Z0-9])'
        if re.search(pattern,text):
            found.add(skill)
    return sorted(found)

PREPROCESSING DATASETS
Skill Vocabulary Size: 329


In [12]:
# ============================================================
# ROLE EXTRACTION
# ============================================================

role_vocab=set(esco.keys())

def extract_roles(text):
    if pd.isna(text):
        return []

    roles = []
    parts = str(text).split(";")
    esco_roles = set(esco.keys())

    for part in parts:
        part = normalize_role(part.strip())

        if part in esco_roles:
            roles.append(part)

    return list(dict.fromkeys(roles))

In [13]:
def choose_best_role(roles, skills):

    if len(roles)==0:
        return ""

    best_role=""
    best_score=0

    skill_set=set(skills)

    for role in roles:

        role_skills=set(esco[role]["skills"])

        score=len(skill_set.intersection(role_skills))

        if score>best_score:

            best_score=score
            best_role=role

    return best_role

In [14]:
# ============================================================
# RESUME DATASET PREPROCESSING
# ============================================================

print("\nCleaning Resume Dataset...")
resumes_clean = resume_df.copy()

# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

if "resume" in resumes_clean.columns:
    resumes_clean = resumes_clean.rename(columns={"resume": "resume_text"})

if "Resume" in resumes_clean.columns:
    resumes_clean = resumes_clean.rename(columns={"Resume": "resume_text"})

# ============================================================
# REMOVE INVALID ROWS
# ============================================================

resumes_clean = resumes_clean.dropna(subset=["resume_text"])
resumes_clean = resumes_clean.drop_duplicates(subset=["resume_text"])
resumes_clean = resumes_clean.reset_index(drop=True)
resumes_clean["resume_text"] = resumes_clean["resume_text"].astype(str)

# ============================================================
# BASIC PREPROCESSING
# ============================================================

resumes_clean["clean_resume"] = resumes_clean["resume_text"].apply(clean_text)
resumes_clean["experience_years"] = resumes_clean["resume_text"].apply(extract_experience)
resumes_clean["sections"] = resumes_clean["resume_text"].apply(extract_sections)

# ============================================================
# SKILL EXTRACTION
# ============================================================

resumes_clean["skills"] = resumes_clean["clean_resume"].apply(extract_skills)
resumes_clean["skill_string"] = resumes_clean["skills"].apply(lambda x: " ".join(x))

# ============================================================
# ROLE EXTRACTION
# ============================================================

if "keywords" in resumes_clean.columns:
    resumes_clean["keywords"] = resumes_clean["keywords"].fillna("")

    resumes_clean["roles"] = resumes_clean["keywords"].apply(extract_roles)

    resumes_clean["role_string"] = resumes_clean["roles"].apply(lambda x: " ".join(x))

    resumes_clean["normalized_role"] = resumes_clean.apply(lambda row: choose_best_role(
            row["roles"], row["skills"]), axis=1)

elif "category" in resumes_clean.columns:
    resumes_clean["normalized_role"] = resumes_clean["category"].fillna("").apply(normalize_role)

else:
    resumes_clean["normalized_role"] = ""

print("Resumes after cleaning :", len(resumes_clean))


Cleaning Resume Dataset...
Resumes after cleaning : 29780


In [15]:
# ============================================================
# JOB DATASET PREPROCESSING
# ============================================================

print("\nCleaning Job Dataset...")

jobs_clean = job_df.copy()
jobs_clean = jobs_clean.sample(n=100000, random_state=42).reset_index(drop=True)
# ============================================================
# STANDARDIZE COLUMN NAMES
# ============================================================

column_mapping = {"Job Description": "job_description",
    "Job Title": "job_title", "position": "job_title", "Role": "role", "Experience": "experience",
    "skills": "raw_skills"}

jobs_clean = jobs_clean.rename(columns=column_mapping)

print("\nAvailable columns:")
print(jobs_clean.columns)

# ============================================================
# REMOVE MISSING VALUES
# ============================================================

jobs_clean = jobs_clean.dropna(subset=["job_description"])
jobs_clean = jobs_clean.reset_index(drop=True)

# ============================================================
# CONVERT TO STRING
# ============================================================

jobs_clean["job_description"] = jobs_clean["job_description"].astype(str)
jobs_clean["job_title"] = jobs_clean["job_title"].astype(str)

# ============================================================
# KEEP ONLY TECH JOBS (OPTIONAL)
# ============================================================

TECH_KEYWORDS = ["developer", "engineer", "software", "web",
    "backend", "frontend", "full stack", "data scientist", "data engineer",
    "data analyst", "machine learning", "artificial intelligence",
    "cloud", "devops", "security", "cyber", "database", "python",
    "java", "network", "qa", "test automation", "ui", "ux"]

pattern = "|".join(re.escape(x) for x in TECH_KEYWORDS)
jobs_clean = jobs_clean[jobs_clean["job_title"].str.lower().str.contains(pattern, na=False)]
jobs_clean = jobs_clean.reset_index(drop=True)

print("Tech jobs:", len(jobs_clean))

# ============================================================
# CLEAN TEXT
# ============================================================

jobs_clean["combined_text"]=(jobs_clean["job_title"].fillna("")+" "+ jobs_clean["Qualifications"].fillna("")+" "+
    jobs_clean["Responsibilities"].fillna("")+" "+ jobs_clean["raw_skills"].fillna("")+" "+ jobs_clean["job_description"].fillna(""))

jobs_clean["clean_job"]=jobs_clean["combined_text"].apply(clean_text)

# ============================================================
# NORMALIZE ROLE
# ============================================================

jobs_clean["normalized_role"] = jobs_clean["job_title"].apply(
    normalize_role
)

# ============================================================
# EXTRACT EXPERIENCE
# ============================================================

if "experience" in jobs_clean.columns:

    jobs_clean["experience_required"] = jobs_clean[
        "experience"
    ].apply(extract_experience)

else:

    jobs_clean["experience_required"] = jobs_clean[
        "job_description"
    ].apply(extract_experience)

# ============================================================
# EXTRACT SECTIONS
# ============================================================

jobs_clean["sections"] = jobs_clean[
    "job_description"
].apply(extract_sections)

# ============================================================
# EXTRACT SKILLS FROM DESCRIPTION
# ============================================================

jobs_clean["skills"] = jobs_clean["clean_job"].apply(extract_skills)

# ============================================================
# MERGE WITH EXISTING SKILLS COLUMN
# ============================================================

if "raw_skills" in jobs_clean.columns:
    jobs_clean["raw_skills"] = jobs_clean["raw_skills"].fillna("").astype(str)
    jobs_clean["skills"] = jobs_clean.apply(

        lambda row: sorted(list(set(row["skills"] + extract_skills(row["raw_skills"])))), axis=1)

# ============================================================
# CREATE SKILL STRING
# ============================================================

jobs_clean["skills"]=jobs_clean["skills"].apply(lambda x: sorted(list(set(x))))

jobs_clean["skill_string"]=jobs_clean["skills"].apply(lambda x:" ".join(x))
print("\nJobs after cleaning :", len(jobs_clean))
print("\nAverage Job Skills :")

print(jobs_clean["skills"].apply(len).mean())
print("\nSample Jobs:")

display(jobs_clean[
        ["job_title", "normalized_role", "experience_required", "skill_string"]].head())


Cleaning Job Dataset...

Available columns:
Index(['Job Id', 'experience', 'Qualifications', 'Salary Range', 'location',
       'Country', 'latitude', 'longitude', 'Work Type', 'Company Size',
       'Job Posting Date', 'Preference', 'Contact Person', 'Contact',
       'job_title', 'role', 'Job Portal', 'job_description', 'Benefits',
       'raw_skills', 'Responsibilities', 'Company', 'Company Profile'],
      dtype='object')
Tech jobs: 25367

Jobs after cleaning : 25367

Average Job Skills :
3.518271770410376

Sample Jobs:


,job_title,normalized_role,experience_required,skill_string
0,Network Engineer,network engineer,None,network security security troubleshooting
1,Network Administrator,network administrator,None,communication firewall incident response netwo...
2,Software Engineer,software engineer,None,communication django express java node.js nosq...
3,Front-End Developer,front end developer,None,communication css html
4,UX/UI Designer,ux/ui designer,None,illustrator photoshop


In [16]:
# ============================================================
# ESCO DATA PREPARATION
# ============================================================

print("\nPreparing ESCO Knowledge Base...")
esco_clean=esco_df.copy()
esco_clean["role"]=esco_clean["occupation"].apply(normalize_role)
esco_clean["skills"]=esco_clean["skills"].apply(lambda x:[normalize_synonyms(i) for i in x])
esco_clean["skill_string"]=esco_clean["skills"].apply(lambda x:" ".join(sorted(set(x))))
print("ESCO Roles :",len(esco_clean))


Preparing ESCO Knowledge Base...
ESCO Roles : 155


In [17]:
# ============================================================
# DATASET STATISTICS
# ============================================================

print("\nAverage Resume Skills :",round(resumes_clean["skills"].apply(len).mean(),2))

print("Average JD Skills :",round(jobs_clean["skills"].apply(len).mean(),2))

print("\nSample Resume")

display(resumes_clean[["normalized_role","experience_years","skill_string"]].head())

print("\nSample Job")

display(jobs_clean[["job_title","experience_required","skill_string"]].head())

print("\nSample ESCO")

display(esco_clean.head())

print("="*60)

print("PREPROCESSING COMPLETED")

print("="*60)


Average Resume Skills : 21.36
Average JD Skills : 3.52

Sample Resume


,normalized_role,experience_years,skill_string
0,database administrator,NaN,backup clustering communication database admin...
1,database administrator,NaN,assembly c c# excel java planning security sql...
2,oracle database administrator,4.0,architecture backup dataguard html linux monit...
3,database administrator,NaN,.net aws backup communication data modeling da...
4,scrum master,NaN,agile architecture backup communication forms ...



Sample Job


,job_title,experience_required,skill_string
0,Network Engineer,None,network security security troubleshooting
1,Network Administrator,None,communication firewall incident response netwo...
2,Software Engineer,None,communication django express java node.js nosq...
3,Front-End Developer,None,communication css html
4,UX/UI Designer,None,illustrator photoshop



Sample ESCO


,occupation,role,skills,skill_string
0,python developer,python developer,"[django, docker, fastapi, flask, git, linux, p...",django docker fastapi flask git linux postgres...
1,java developer,java developer,"[docker, git, hibernate, java, maven, microser...",docker git hibernate java maven microservices ...
2,.net developer,.net developer,"[.net, asp.net, azure, c#, entity framework, g...",.net asp.net azure c# entity framework git sql...
3,frontend developer,frontend developer,"[angular, css, git, html, javascript, react, t...",angular css git html javascript react typescri...
4,backend developer,backend developer,"[docker, java, kubernetes, node.javascript, py...",docker java kubernetes node.javascript python ...


PREPROCESSING COMPLETED


In [18]:
print(
    jobs_clean["normalized_role"]
    .value_counts()
    .head(30)
)

normalized_role
ux/ui designer                 3070
software engineer              1957
network engineer               1499
software tester                1319
network administrator          1118
data analyst                   1028
mechanical engineer             892
ui developer                    882
civil engineer                  823
network security specialist     676
java developer                  676
qa analyst                      671
data engineer                   656
chemical engineer               647
front end developer             642
aerospace engineer              638
ux researcher                   634
structural engineer             627
database administrator          625
electrical engineer             625
software architect              618
web developer                   605
process engineer                472
web designer                    471
front end engineer              467
back end developer              462
database developer              443
network tech

In [19]:
print(
    jobs_clean["skills"]
    .apply(len)
    .describe()
)

count    25367.000000
mean         3.518272
std          2.695771
min          0.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         13.000000
Name: skills, dtype: float64


## Module 5: Dataset Splitting & Training Pair Generation

To avoid **data leakage**, we split the resumes and job descriptions at the document level *prior* to generating training pairs. Once split, we create three distinct alignment pair types for training:
1. **Resume $\leftrightarrow$ Job Description**: Core semantic mapping.
2. **Resume $\leftrightarrow$ Job Role**: Map resume text directly to normalized job titles.
3. **Resume $\leftrightarrow$ Skills**: Connect profile experiences to isolated technical tags.

We also select a 1-to-1 matching set of resumes and job descriptions from the validation split to evaluate semantic retrieval performance.

In [20]:
# ============================================================
# 5. TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

train_resumes,val_resumes=train_test_split(
    resumes_clean,
    test_size=0.2,
    random_state=SEED,
    shuffle=True)

train_jobs,val_jobs=train_test_split(
    jobs_clean,
    test_size=0.2,
    random_state=SEED,
    shuffle=True)

print("Training resumes :",len(train_resumes))
print("Validation resumes :",len(val_resumes))

print("Training jobs :",len(train_jobs))
print("Validation jobs :",len(val_jobs))

Training resumes : 23824
Validation resumes : 5956
Training jobs : 20293
Validation jobs : 5074


In [21]:
GENERIC_SKILLS = {
    "communication",
    "planning",
    "documentation",
    "leadership",
    "excel",
    "problem solving",
    "teamwork",
    "quality assurance",
    "monitoring"}

In [22]:
# ============================================================
# ROLE INFERENCE
# ============================================================

def infer_role(skills):
    skills = {s.lower() for s in skills if s.lower() not in GENERIC_SKILLS}

    if len(skills) == 0:
        return ""

    best_role = ""
    best_score = 0

    for _, row in esco_clean.iterrows():
        role_skills = {s.lower() for s in row["skills"] if s.lower() not in GENERIC_SKILLS}

        intersection = len(skills & role_skills)
        union = len(skills | role_skills)

        if union == 0:
            continue

        score = intersection / union

        if score > best_score:

            best_score = score
            best_role = row["role"]

    return best_role

In [23]:
def role_similarity(skills, role):

    if role == "":
        return 0

    resume_skills = set(skills)

    esco_match = esco_clean[esco_clean["role"] == role]

    if len(esco_match) == 0:
        return 0

    role_skills = set(esco_match.iloc[0]["skills"])

    intersection = len(resume_skills & role_skills)
    union = len(resume_skills | role_skills)

    if union == 0:
        return 0

    return intersection / union

In [24]:
train_resumes["predicted_role"] = train_resumes["normalized_role"]
mask = train_resumes["predicted_role"] == ""

train_resumes.loc[mask, "predicted_role"] = (train_resumes.loc[mask, "skills"].apply(infer_role))
val_resumes["predicted_role"] = val_resumes["normalized_role"]

mask = val_resumes["predicted_role"] == ""
val_resumes.loc[mask, "predicted_role"] = (val_resumes.loc[mask, "skills"].apply(infer_role))

In [25]:
# ============================================================
# POSITIVE PAIR GENERATION
# ============================================================

def generate_positive_pairs(resume_df, job_df, esco_df):
    pairs = set()
    for _, resume in resume_df.iterrows():

        resume_text = resume["clean_resume"]
        role = resume["predicted_role"]
        resume_skills = resume["skills"]

        confidence = role_similarity(resume_skills, role)

        if confidence < 0.15:
            continue

        if len(resume_skills) == 0:
            continue

        skill_string = " ".join(resume_skills)

        # ------------------------------------------------
        # Resume ↔ Role
        # ------------------------------------------------

        if role != "":
            pairs.add((resume_text, role))

        # ------------------------------------------------
        # Resume ↔ Skills
        # ------------------------------------------------

        if skill_string != "":
            pairs.add((resume_text, skill_string))

        # ------------------------------------------------
        # Resume ↔ ESCO skills
        # ------------------------------------------------

        esco_match = esco_df[esco_df["role"] == role]
        if len(esco_match) > 0:
            esco_skill_string = esco_match.iloc[0]["skill_string"]

            if esco_skill_string != "":
                pairs.add((resume_text, esco_skill_string))

        # ------------------------------------------------
        # Resume ↔ Matching jobs
        # ------------------------------------------------

        matches = job_df[(job_df["normalized_role"] == role) &
                  (job_df["skills"].apply(lambda x: len(set(x) & set(resume["skills"])) >= 2))]

        if len(matches) > 0:

            sample = matches.sample(n=min(3, len(matches)), random_state=SEED)

            for _, jd in sample.iterrows():
                job_text = jd["clean_job"]

                if len(job_text) > 50:
                    pairs.add((resume_text, job_text))

    return list(pairs)

In [27]:
train_pairs=generate_positive_pairs(train_resumes, train_jobs, esco_clean)
print("Training pairs :",len(train_pairs))

Training pairs : 55050


In [28]:
DEGREES = [
    "bca",
    "bsc",
    "btech",
    "mca",
    "msc",
    "mba",
    "phd",
    "bba"
]

pattern = r"\b(" + "|".join(DEGREES) + r")\b"

jobs_clean["clean_job"] = jobs_clean["clean_job"].str.replace(
    pattern,
    "",
    regex=True
)

In [29]:
# ============================================================
# VALIDATION SET
# ============================================================

validation_queries=[]
validation_docs=[]
for _,resume in val_resumes.iterrows():
    role=resume["predicted_role"]
    matches=val_jobs[val_jobs["normalized_role"]==role]

    if len(matches)==0:
        continue

    validation_queries.append(
        resume["clean_resume"])

    validation_docs.append(
        matches.iloc[0]["clean_job"])

print("Validation Queries :",len(validation_queries))
print("Validation Docs :",len(validation_docs))

Validation Queries : 2172
Validation Docs : 2172


In [30]:
print("="*60)

print("Sample Training Pair")

print()

print(train_pairs[0][0][:300])

print()

print("-----------------------------")

print()

print(train_pairs[0][1][:300])

print()

print("="*60)

Sample Training Pair

python program  span class  hl  python /span  program san jose  ca work experience python program hmm hidden markov model  on path based map matching  northwestern university   evanston  il march 2016 to november 2016  used python verified the positive impact of hidden markov model on increasing pre

-----------------------------

database design mysql pl/sql postgresql sql stored procedures triggers



In [31]:
from collections import Counter

counter = Counter()

for _, role in train_pairs:

    counter[role] += 1

print(counter.most_common(20))

[('web developer', 1545), ('bootstrap css git html javascript node.javascript react', 1545), ('system administrator', 1012), ('active directory bash linux powershell windows server', 1012), ('docker git hibernate java maven microservices mysql spring spring boot', 964), ('java developer', 964), ('web developer mba design and code user interfaces for websites  ensuring a seamless and visually appealing user experience. collaborate with ux designers to optimize user journeys. ensure cross browser compatibility and responsive design. html  css  javascript frontend frameworks  e.g.  react  angular  user experience  ux  frontend web developers design and implement user interfaces for websites  ensuring they are visually appealing and user friendly. they collaborate with designers and backend developers to create seamless web experiences for users.', 943), ('web developer ba design and code user interfaces for websites  ensuring a seamless and visually appealing user experience. collaborate 

## Module 6: Retrieval Evaluation Suite

We implement custom metrics instead of generic classification scores to evaluate the model's retrieval capability:
- **Precision@1 & Precision@5**: Probability that the correct match is in the top-1 or top-5 recommendations.
- **Recall@5**: Probability of retrieving the correct item within 5 attempts.
- **Mean Reciprocal Rank (MRR)**: Evaluates the rank order of the target match.
- **Mean Cosine Similarity** of matched pairs.

The evaluator runs batch-level tokenization using standard PyTorch parameters.

In [34]:
# ============================================================
# 6. RETRIEVAL EVALUATOR CLASS
# ============================================================

class RetrievalEvaluator:
    """
    Evaluates resume-to-job retrieval using standard IR-style metrics.

    Validation setup:
        - One resume/query
        - One known relevant job/document for each query

    Metrics:
        - Mean cosine similarity
        - Hit / Precision@1
        - Hit / Recall@5
        - MRR
    """

    def __init__(
        self,
        queries: List[str],
        documents: List[str],
        device: torch.device
    ):
        self.queries = queries
        self.documents = documents
        self.device = device

    def _encode_texts(
        self,
        model: SentenceTransformer,
        texts: List[str],
        batch_size: int = 32
    ) -> np.ndarray:

        if len(texts) == 0:
            return np.empty((0, 0))

        embeddings = model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

        return embeddings

    def evaluate(
        self,
        model: SentenceTransformer
    ) -> Dict[str, float]:

        # ----------------------------------------------------
        # Empty validation set check
        # ----------------------------------------------------

        if len(self.queries) == 0 or len(self.documents) == 0:

            return {
                "cosine_similarity_mean": 0.0,
                "precision_at_1": 0.0,
                "precision_at_5": 0.0,
                "recall_at_5": 0.0,
                "mrr": 0.0
            }

        # ----------------------------------------------------
        # Make sure query/document counts match
        # ----------------------------------------------------

        if len(self.queries) != len(self.documents):

            raise ValueError(
                f"Number of validation queries ({len(self.queries)}) "
                f"does not match number of validation documents "
                f"({len(self.documents)})."
            )

        # ----------------------------------------------------
        # Generate embeddings
        # ----------------------------------------------------

        q_emb = self._encode_texts(
            model,
            self.queries
        )

        d_emb = self._encode_texts(
            model,
            self.documents
        )

        # ----------------------------------------------------
        # Cosine similarity matrix
        #
        # Because embeddings are normalized,
        # dot product = cosine similarity.
        # ----------------------------------------------------

        sim_matrix = np.dot(
            q_emb,
            d_emb.T
        )

        num_queries = len(self.queries)

        reciprocal_ranks = []

        hits_at_1 = 0
        hits_at_5 = 0

        diagonal_similarities = []

        # ----------------------------------------------------
        # Calculate retrieval metrics
        # ----------------------------------------------------

        for i in range(num_queries):

            scores = sim_matrix[i]

            # Rank documents from highest similarity
            # to lowest similarity
            sorted_indices = np.argsort(scores)[::-1]

            # The validation construction assumes that
            # document i is the relevant document for
            # query i.
            relevant_position = np.where(
                sorted_indices == i
            )[0]

            if len(relevant_position) == 0:
                continue

            rank = int(relevant_position[0]) + 1

            reciprocal_ranks.append(
                1.0 / rank
            )

            # Similarity between the query and its
            # known relevant document
            diagonal_similarities.append(
                sim_matrix[i, i]
            )

            # Hit@1
            if rank == 1:
                hits_at_1 += 1

            # Hit@5 / Recall@5
            if rank <= 5:
                hits_at_5 += 1

        # ----------------------------------------------------
        # Final metrics
        # ----------------------------------------------------

        if len(reciprocal_ranks) == 0:

            return {
                "cosine_similarity_mean": 0.0,
                "precision_at_1": 0.0,
                "precision_at_5": 0.0,
                "recall_at_5": 0.0,
                "mrr": 0.0
            }

        return {

            # Mean similarity between each resume
            # and its known relevant job
            "cosine_similarity_mean": float(
                np.mean(diagonal_similarities)
            ),

            # With one relevant document per query,
            # this is effectively Hit@1
            "precision_at_1": float(
                hits_at_1 / num_queries
            ),

            # Kept for compatibility with the original
            # evaluator. With one relevant document per
            # query, this is better interpreted as Hit@5.
            "precision_at_5": float(
                hits_at_5 / num_queries
            ),

            # Recall@5 = whether the relevant document
            # appeared in the top 5 results.
            "recall_at_5": float(
                hits_at_5 / num_queries
            ),

            # Mean Reciprocal Rank
            "mrr": float(
                np.mean(reciprocal_ranks)
            )
        }


# ============================================================
# INITIALIZE VALIDATION EVALUATOR
# ============================================================

evaluator = RetrievalEvaluator(
    validation_queries,
    validation_docs,
    device
)

print("Retrieval evaluator initialized successfully.")

print("Validation queries :", len(validation_queries))
print("Validation documents :", len(validation_docs))

Retrieval evaluator initialized successfully.
Validation queries : 2172
Validation documents : 2172


## 7. Sentence Transformer Model Setup

This module loads the pretrained **Sentence Transformer** model that will be used for resume–job semantic matching.

The model used is:

**`sentence-transformers/all-mpnet-base-v2`**

The model converts resumes, job descriptions, skills, and role descriptions into dense numerical **embeddings**. These embeddings allow the system to measure semantic similarity between a candidate's resume and job requirements.

In this module:

- The pretrained `all-mpnet-base-v2` model is loaded.
- The model is placed on the available computation device (`CUDA` when available).
- The embedding dimension is verified.
- No fine-tuning or training is performed yet.

The loaded model produces **768-dimensional embeddings**, which will later be used for semantic retrieval and resume–job matching.

The pretrained model will serve as the starting point for the subsequent fine-tuning stage using the generated positive resume–job pairs.

In [35]:
# ============================================================
# 7. SENTENCE TRANSFORMER MODEL SETUP
# ============================================================

from sentence_transformers import SentenceTransformer

# ------------------------------------------------------------
# Load base Sentence Transformer model
# ------------------------------------------------------------

print("=" * 60)
print("LOADING SENTENCE TRANSFORMER MODEL")
print("=" * 60)

model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print()
print("Model loaded successfully.")
print("Model :", MODEL_NAME)
print("Device :", device)

# ------------------------------------------------------------
# Display embedding dimension
# ------------------------------------------------------------

embedding_dimension = model.get_sentence_embedding_dimension()

print("Embedding dimension :", embedding_dimension)

print("=" * 60)

LOADING SENTENCE TRANSFORMER MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Model loaded successfully.
Model : sentence-transformers/all-mpnet-base-v2
Device : cuda
Embedding dimension : 768


## 8. Sentence Transformer Fine-Tuning

The positive training pairs generated in Module 5 are used to fine-tune the pretrained `all-mpnet-base-v2` Sentence Transformer.

Each training pair consists of:

- **Anchor:** a resume
- **Positive:** a related role, skill description, ESCO skill description, or matching job description

The model is fine-tuned using **Multiple Negatives Ranking Loss (MNRL)**. Within each training batch, the matching document is treated as the positive example for its corresponding resume, while the other documents in the batch act as negative examples.

The objective is to improve the embedding space so that semantically relevant resumes and job-related documents have higher similarity than unrelated documents.

This is the main model-training stage of the pipeline.

The validation set created earlier is kept separate from training and will be used after fine-tuning to evaluate retrieval performance.

In [36]:
# ============================================================
# 8. SENTENCE TRANSFORMER FINE-TUNING
# ============================================================

from torch.utils.data import DataLoader
from sentence_transformers import InputExample
from sentence_transformers.losses import MultipleNegativesRankingLoss


# ============================================================
# VERIFY EXISTING TRAINING PAIRS
# ============================================================

print("=" * 60)
print("STARTING SENTENCE TRANSFORMER FINE-TUNING")
print("=" * 60)

print("Existing training pairs :", len(train_pairs))

if len(train_pairs) == 0:
    raise ValueError(
        "train_pairs is empty. "
        "Run the positive pair generation module first."
    )


# ============================================================
# CONVERT EXISTING PAIRS TO INPUT EXAMPLES
# ============================================================

train_examples = []

for anchor, positive in train_pairs:

    if not isinstance(anchor, str):
        continue

    if not isinstance(positive, str):
        continue

    anchor = anchor.strip()
    positive = positive.strip()

    if not anchor or not positive:
        continue

    train_examples.append(
        InputExample(
            texts=[anchor, positive]
        )
    )


print("Valid training examples :", len(train_examples))

if len(train_examples) == 0:
    raise ValueError(
        "No valid training examples were created."
    )


# ============================================================
# CREATE TRAINING DATALOADER
# ============================================================

BATCH_SIZE = 16

train_dataloader = DataLoader(
    train_examples,
    shuffle=True,
    batch_size=BATCH_SIZE
)


# ============================================================
# CREATE TRAINING LOSS
# ============================================================

train_loss = MultipleNegativesRankingLoss(model)


# ============================================================
# TRAINING PARAMETERS
# ============================================================

NUM_EPOCHS = 1

WARMUP_STEPS = max(
    1,
    int(len(train_dataloader) * NUM_EPOCHS * 0.1)
)

OUTPUT_PATH = "./resume_job_mpnet_finetuned"


# ============================================================
# DISPLAY TRAINING CONFIGURATION
# ============================================================

print()
print("Training configuration")
print("-" * 60)
print("Model              :", MODEL_NAME)
print("Device             :", device)
print("Training examples  :", len(train_examples))
print("Batch size         :", BATCH_SIZE)
print("Epochs             :", NUM_EPOCHS)
print("Warmup steps       :", WARMUP_STEPS)
print("Output path        :", OUTPUT_PATH)
print()


# ============================================================
# FINE-TUNE MODEL
# ============================================================

model.fit(
    train_objectives=[
        (train_dataloader, train_loss)
    ],
    epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    output_path=OUTPUT_PATH,
    show_progress_bar=True
)


# ============================================================
# TRAINING COMPLETED
# ============================================================

print()
print("=" * 60)
print("FINE-TUNING COMPLETED SUCCESSFULLY")
print("=" * 60)

print("Fine-tuned model saved to:")
print(OUTPUT_PATH)

STARTING SENTENCE TRANSFORMER FINE-TUNING
Existing training pairs : 55050
Valid training examples : 55050

Training configuration
------------------------------------------------------------
Model              : sentence-transformers/all-mpnet-base-v2
Device             : cuda
Training examples  : 55050
Batch size         : 16
Epochs             : 1
Warmup steps       : 344
Output path        : ./resume_job_mpnet_finetuned



Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/3441 [00:00<?, ?it/s]


FINE-TUNING COMPLETED SUCCESSFULLY
Fine-tuned model saved to:
./resume_job_mpnet_finetuned


# ============================================================
# 9. FINE-TUNED MODEL EVALUATION
# ============================================================

This module evaluates the fine-tuned Sentence Transformer on the
validation retrieval dataset.

The model is evaluated against the same validation queries and
documents that were prepared earlier.

The evaluation measures:

- Mean cosine similarity
- Precision@1
- Precision@5
- Recall@5
- Mean Reciprocal Rank (MRR)

These metrics are used to determine whether fine-tuning improved
the model's ability to retrieve relevant job-related documents
for a given resume.

The evaluation is performed using the fine-tuned model saved in:

`./resume_job_mpnet_finetuned`

No additional training is performed in this module.

In [37]:
# ============================================================
# 9. FINE-TUNED MODEL EVALUATION
# ============================================================
from sentence_transformers import SentenceTransformer

print("=" * 60)
print("LOADING FINE-TUNED MODEL FOR EVALUATION")
print("=" * 60)
FINETUNED_MODEL_PATH = "./resume_job_mpnet_finetuned"

finetuned_model = SentenceTransformer(FINETUNED_MODEL_PATH, device=str(device))

print("Fine-tuned model loaded successfully.")
print("Model path :", FINETUNED_MODEL_PATH)
print("Device     :", device)
print()

# ============================================================
# VALIDATION DATA CHECK
# ============================================================

print("=" * 60)
print("VALIDATION DATA")
print("=" * 60)

print("Validation queries    :", len(validation_queries))
print("Validation documents  :", len(validation_docs))

if len(validation_queries) == 0:
    raise ValueError("Validation queries are empty.")

if len(validation_docs) == 0:
    raise ValueError("Validation documents are empty.")

if len(validation_queries) != len(validation_docs):
    raise ValueError("Validation queries and documents must have the same length.")

print("Validation data check passed.")
print()

# ============================================================
# CREATE EVALUATOR FOR FINE-TUNED MODEL
# ============================================================

finetuned_evaluator = RetrievalEvaluator(validation_queries, validation_docs, device)

# ============================================================
# EVALUATE FINE-TUNED MODEL
# ============================================================

print("=" * 60)
print("EVALUATING FINE-TUNED MODEL")
print("=" * 60)
finetuned_metrics = finetuned_evaluator.evaluate(finetuned_model)

# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 60)
print("FINE-TUNED MODEL RESULTS")
print("=" * 60)

print(f"Mean Cosine Similarity : " f"{finetuned_metrics['cosine_similarity_mean']:.4f}")
print(f"Precision@1            : " f"{finetuned_metrics['precision_at_1']:.4f}")
print(f"Precision@5            : " f"{finetuned_metrics['precision_at_5']:.4f}")
print(f"Recall@5               : " f"{finetuned_metrics['recall_at_5']:.4f}")
print(f"MRR                    : " f"{finetuned_metrics['mrr']:.4f}")

print("=" * 60)

LOADING FINE-TUNED MODEL FOR EVALUATION
Fine-tuned model loaded successfully.
Model path : ./resume_job_mpnet_finetuned
Device     : cuda

VALIDATION DATA
Validation queries    : 2172
Validation documents  : 2172
Validation data check passed.

EVALUATING FINE-TUNED MODEL

FINE-TUNED MODEL RESULTS
Mean Cosine Similarity : 0.4649
Precision@1            : 0.0037
Precision@5            : 0.0226
Recall@5               : 0.0226
MRR                    : 0.0201


# ============================================================
# 10. ROLE-BASED RETRIEVAL EVALUATION
# ============================================================

The previous evaluation treated the document at the same index as
the query as the only correct document.

For resume-to-job matching, this is too restrictive because a
resume can be relevant to multiple job descriptions with the same
or closely related role.

This module therefore performs a role-aware retrieval evaluation.

For each validation resume:

1. The fine-tuned model embeds the resume.
2. All validation job descriptions are ranked by cosine similarity.
3. A retrieved job is considered relevant when its normalized role
   matches the resume's inferred role.
4. Retrieval quality is measured using:
   - Recall@1
   - Recall@5
   - Recall@10
   - MRR

This provides a more meaningful evaluation of whether the model
retrieves jobs belonging to the appropriate role.

No model training is performed in this module.

In [40]:
# ============================================================
# 10. ROLE-BASED RETRIEVAL EVALUATION
# ============================================================

import numpy as np


print("=" * 60)
print("ROLE-BASED RETRIEVAL EVALUATION")
print("=" * 60)


# ============================================================
# BUILD VALIDATION QUERIES AND ROLES
# ============================================================

validation_queries_role = []
validation_roles = []

for _, resume in val_resumes.iterrows():

    role = str(
        resume["predicted_role"]
    ).strip().lower()

    # Skip resumes without an inferred role
    if not role:
        continue

    validation_queries_role.append(
        resume["clean_resume"]
    )

    validation_roles.append(
        role
    )


# ============================================================
# BUILD VALIDATION JOB DOCUMENTS AND ROLES
# ============================================================

validation_job_docs = []
validation_job_roles = []

for _, job in val_jobs.iterrows():

    job_text = str(
        job["clean_job"]
    ).strip()

    job_role = str(
        job["normalized_role"]
    ).strip().lower()

    # Skip jobs without usable text or role
    if not job_text or not job_role:
        continue

    validation_job_docs.append(
        job_text
    )

    validation_job_roles.append(
        job_role
    )


# ============================================================
# DISPLAY DATASET SIZES
# ============================================================

print(
    "Validation resumes with roles :",
    len(validation_queries_role)
)

print(
    "Validation jobs with roles    :",
    len(validation_job_docs)
)


# ============================================================
# ALIGNMENT CHECK
# ============================================================

if len(validation_queries_role) != len(validation_roles):

    raise ValueError(
        "Validation queries and validation roles "
        "have different lengths."
    )


if len(validation_job_docs) != len(validation_job_roles):

    raise ValueError(
        "Validation job documents and validation job roles "
        "have different lengths."
    )


if len(validation_queries_role) == 0:

    raise ValueError(
        "No validation resumes with inferred roles were found."
    )


if len(validation_job_docs) == 0:

    raise ValueError(
        "No validation jobs with valid roles were found."
    )


print()
print("Validation data alignment check passed.")


# ============================================================
# ENCODE VALIDATION RESUMES
# ============================================================

print()
print("Encoding validation resumes...")

query_embeddings = finetuned_model.encode(
    validation_queries_role,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# ============================================================
# ENCODE VALIDATION JOBS
# ============================================================

print()
print("Encoding validation jobs...")

job_embeddings = finetuned_model.encode(
    validation_job_docs,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


# ============================================================
# SIMILARITY MATRIX
# ============================================================

similarity_matrix = np.matmul(
    query_embeddings,
    job_embeddings.T
)


print()
print(
    "Similarity matrix shape :",
    similarity_matrix.shape
)


# ============================================================
# ROLE-BASED RETRIEVAL METRICS
# ============================================================

hits_at_1 = 0
hits_at_5 = 0
hits_at_10 = 0

reciprocal_ranks = []

evaluated_queries = 0


# ============================================================
# EVALUATE EACH RESUME
# ============================================================

for query_index in range(
    len(validation_queries_role)
):

    query_role = validation_roles[
        query_index
    ]

    scores = similarity_matrix[
        query_index
    ]

    # Rank all jobs by similarity
    ranked_indices = np.argsort(
        scores
    )[::-1]


    # --------------------------------------------------------
    # Find first job with the correct role
    # --------------------------------------------------------

    relevant_rank = None

    for rank, job_index in enumerate(
        ranked_indices,
        start=1
    ):

        job_role = validation_job_roles[
            job_index
        ]

        if job_role == query_role:

            relevant_rank = rank

            break


    # --------------------------------------------------------
    # Skip if no matching role exists
    # --------------------------------------------------------

    if relevant_rank is None:

        continue


    evaluated_queries += 1


    # --------------------------------------------------------
    # Recall@1
    # --------------------------------------------------------

    if relevant_rank <= 1:

        hits_at_1 += 1


    # --------------------------------------------------------
    # Recall@5
    # --------------------------------------------------------

    if relevant_rank <= 5:

        hits_at_5 += 1


    # --------------------------------------------------------
    # Recall@10
    # --------------------------------------------------------

    if relevant_rank <= 10:

        hits_at_10 += 1


    # --------------------------------------------------------
    # Reciprocal Rank
    # --------------------------------------------------------

    reciprocal_ranks.append(
        1.0 / relevant_rank
    )


# ============================================================
# CHECK EVALUATION
# ============================================================

if evaluated_queries == 0:

    raise ValueError(
        "No validation queries had a matching role "
        "in the validation job set."
    )


# ============================================================
# CALCULATE FINAL METRICS
# ============================================================

role_recall_at_1 = (
    hits_at_1 /
    evaluated_queries
)

role_recall_at_5 = (
    hits_at_5 /
    evaluated_queries
)

role_recall_at_10 = (
    hits_at_10 /
    evaluated_queries
)

role_mrr = np.mean(
    reciprocal_ranks
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print()
print("=" * 60)
print("ROLE-BASED RETRIEVAL RESULTS")
print("=" * 60)

print(
    f"Evaluated queries : "
    f"{evaluated_queries}"
)

print(
    f"Recall@1          : "
    f"{role_recall_at_1:.4f}"
)

print(
    f"Recall@5          : "
    f"{role_recall_at_5:.4f}"
)

print(
    f"Recall@10         : "
    f"{role_recall_at_10:.4f}"
)

print(
    f"MRR               : "
    f"{role_mrr:.4f}"
)

print("=" * 60)

ROLE-BASED RETRIEVAL EVALUATION
Validation resumes with roles : 5896
Validation jobs with roles    : 5074

Validation data alignment check passed.

Encoding validation resumes...


Batches:   0%|          | 0/185 [00:00<?, ?it/s]


Encoding validation jobs...


Batches:   0%|          | 0/159 [00:00<?, ?it/s]


Similarity matrix shape : (5896, 5074)

ROLE-BASED RETRIEVAL RESULTS
Evaluated queries : 2172
Recall@1          : 0.5649
Recall@5          : 0.5755
Recall@10         : 0.5902
MRR               : 0.5741
